# Proyecto final: EDA de reservas de hotel
## Notebook 01 · Carga, limpieza y unión de los datos
**Alumno:** Carlos Luque Camacho

En este notebook preparo el conjunto de datos final del proyecto. Trabajo con tres fuentes distintas:

1. **Reservas de hotel** (Kaggle, *Hotel Booking Demand*): reservas de un hotel de ciudad (Lisboa) y un resort (Algarve) entre julio de 2015 y agosto de 2017.
2. **Banco Mundial**: PIB per cápita, región y nivel de renta de cada país.
3. **Open-Meteo**: tiempo diario en Lisboa y en el Algarve.

Pasos: cargar y explorar cada fuente, limpiar las reservas, preparar las otras dos fuentes, unirlo todo y guardar el resultado en `data/processed`.

In [1]:
# Librerías
import pandas as pd
import numpy as np

# Rutas (el notebook está en la carpeta notebooks, por eso subo un nivel con ../)
RUTA_RAW = "../data/raw/"
RUTA_PROCESSED = "../data/processed/"

# Para ver todas las columnas al mostrar un DataFrame
pd.set_option("display.max_columns", None)

## 1. Carga de los datos

In [2]:
# Fuente 1: reservas
reservas = pd.read_csv(RUTA_RAW + "hotel_bookings.csv")

# Fuente 2: Banco Mundial. El archivo del PIB tiene 4 filas de cabecera que no son datos, por eso skiprows=4
pib = pd.read_csv(RUTA_RAW + "API_NY.GDP.PCAP.CD_DS2_en_csv_v2_260096.csv", skiprows=4)
paises = pd.read_csv(RUTA_RAW + "Metadata_Country_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_260096.csv")

# Fuente 3: tiempo. Los CSV de Open-Meteo tienen 3 filas al principio con la ubicación, por eso skiprows=3
tiempo_lisboa = pd.read_csv(RUTA_RAW + "tiempo_lisboa.csv.csv", skiprows=3)
tiempo_algarve = pd.read_csv(RUTA_RAW + "tiempo_algarve.csv.csv", skiprows=3)

print("Reservas:", reservas.shape)
print("PIB:", pib.shape)
print("Países:", paises.shape)
print("Tiempo Lisboa:", tiempo_lisboa.shape)
print("Tiempo Algarve:", tiempo_algarve.shape)

Reservas: (119390, 32)
PIB: (266, 71)
Países: (265, 6)
Tiempo Lisboa: (793, 5)
Tiempo Algarve: (793, 5)


## 2. Exploración inicial de las reservas

In [3]:
reservas.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [4]:
reservas.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [5]:
# Nulos por columna (solo las que tienen alguno)
nulos = reservas.isna().sum()
nulos[nulos > 0]

children         4
country        488
agent        16340
company     112593
dtype: int64

In [6]:
# Filas duplicadas
print("Filas duplicadas:", reservas.duplicated().sum())

Filas duplicadas: 31994


In [7]:
# Estadísticos de las columnas numéricas
reservas.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119386.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,103050.000000,6797.000000,119390.000000,119390.000000,119390.000000,119390.000000
mean,0.370416,104.011416,2016.156554,27.165173,15.798241,0.927599,2.500302,1.856403,0.103890,0.007949,0.031912,0.087118,0.137097,0.221124,86.693382,189.266735,2.321149,101.831122,0.062518,0.571363
std,0.482918,106.863097,0.707476,13.605138,8.780829,0.998613,1.908286,0.579261,0.398561,0.097436,0.175767,0.844336,1.497437,0.652306,110.774548,131.655015,17.594721,50.535790,0.245291,0.792798
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,0.000000,-6.380000,0.000000,0.000000
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,62.000000,0.000000,69.290000,0.000000,0.000000
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,179.000000,0.000000,94.575000,0.000000,0.000000
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,229.000000,270.000000,0.000000,126.000000,0.000000,1.000000
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,1.000000,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000


In [8]:
# Valores de algunas columnas categóricas para ver si hay valores raros
for columna in ["hotel", "meal", "market_segment", "distribution_channel", "deposit_type", "customer_type", "reservation_status"]:
    print(columna, reservas[columna].unique())

hotel <StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str
meal <StringArray>
['BB', 'FB', 'HB', 'SC', 'Undefined']
Length: 5, dtype: str
market_segment <StringArray>
[       'Direct',     'Corporate',     'Online TA', 'Offline TA/TO',
 'Complementary',        'Groups',     'Undefined',      'Aviation']
Length: 8, dtype: str
distribution_channel <StringArray>
['Direct', 'Corporate', 'TA/TO', 'Undefined', 'GDS']
Length: 5, dtype: str
deposit_type <StringArray>
['No Deposit', 'Refundable', 'Non Refund']
Length: 3, dtype: str
customer_type <StringArray>
['Transient', 'Contract', 'Transient-Party', 'Group']
Length: 4, dtype: str
reservation_status <StringArray>
['Check-Out', 'Canceled', 'No-Show']
Length: 3, dtype: str


**Lo que encuentro en la exploración:**
- Hay muchas filas duplicadas.
- Nulos en `children` (4), `country`, `agent` y sobre todo en `company` (casi el 95 %).
- `adr` (precio medio por noche) tiene valores negativos y un máximo de 5.400 €, que es un error claro.
- En `meal` aparece el valor `Undefined`.
- La fecha de llegada está repartida en tres columnas (año, mes en texto y día).

## 3. Limpieza de las reservas

In [9]:
# Copio el DataFrame para no tocar los datos originales
df = reservas.copy()
filas_iniciales = len(df)

### 3.1 Duplicados
El dataset no tiene un identificador de reserva, así que una fila exactamente igual a otra en sus 32 columnas
(mismo cliente, mismas fechas, mismo precio...) la considero un registro repetido y la elimino.

In [10]:
df = df.drop_duplicates()
print("Duplicados eliminados:", filas_iniciales - len(df))
print("Filas que quedan:", len(df))

Duplicados eliminados: 31994
Filas que quedan: 87396


### 3.2 Nulos

In [11]:
# children: solo hay 4 nulos, lo más lógico es que no viajaran niños
df["children"] = df["children"].fillna(0).astype(int)

# country: no sé de dónde es el cliente, así que lo marco como desconocido
df["country"] = df["country"].fillna("Desconocido")

# agent: el nulo significa que la reserva no se hizo a través de agencia
df["tiene_agencia"] = df["agent"].notna()

# company: casi todo son nulos, así que no aporta como identificador.
# Me quedo solo con la información de si la reserva la hizo una empresa
df["reserva_empresa"] = df["company"].notna()

# Elimino las columnas de identificadores, ya que solo son números de agencia y empresa
df = df.drop(columns=["agent", "company"])

df.isna().sum().sum()

np.int64(0)

### 3.3 Valores incorrectos

In [12]:
# Reservas sin ningún huésped (0 adultos, 0 niños y 0 bebés): no tienen sentido
sin_huespedes = (df["adults"] + df["children"] + df["babies"]) == 0
print("Reservas sin huéspedes:", sin_huespedes.sum())
df = df[~sin_huespedes]

# Reservas con 0 noches: no son estancias reales de hotel
df["total_noches"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
cero_noches = df["total_noches"] == 0
print("Reservas con 0 noches:", cero_noches.sum())
df = df[~cero_noches]

# Precio medio por noche (adr) negativo o absurdamente alto
adr_incorrecto = (df["adr"] < 0) | (df["adr"] > 1000)
print("Reservas con adr incorrecto:", adr_incorrecto.sum())
df = df[~adr_incorrecto]

Reservas sin huéspedes: 166
Reservas con 0 noches: 591
Reservas con adr incorrecto: 2


In [13]:
# En meal, 'Undefined' y 'SC' significan lo mismo (sin comidas), según la documentación del dataset
df["meal"] = df["meal"].replace("Undefined", "SC")

# Códigos de país que no siguen el estándar ISO-3 que usa el Banco Mundial
# CN -> CHN (China) y TMP -> TLS (Timor Oriental)
df["country"] = df["country"].replace({"CN": "CHN", "TMP": "TLS"})

print("Filas tras la limpieza:", len(df))

Filas tras la limpieza: 86637


### 3.4 Fechas
La fecha de llegada viene en tres columnas. La junto en una sola columna de tipo fecha.

In [14]:
# El mes viene en texto en inglés, lo paso a número con un diccionario
meses = {"January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6,
         "July": 7, "August": 8, "September": 9, "October": 10, "November": 11, "December": 12}
df["mes_llegada"] = df["arrival_date_month"].map(meses)

df["fecha_llegada"] = pd.to_datetime(
    dict(year=df["arrival_date_year"], month=df["mes_llegada"], day=df["arrival_date_day_of_month"])
)
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])

df[["arrival_date_year", "arrival_date_month", "arrival_date_day_of_month", "fecha_llegada"]].head()

,arrival_date_year,arrival_date_month,arrival_date_day_of_month,fecha_llegada
2,2015,July,1,2015-07-01
3,2015,July,1,2015-07-01
4,2015,July,1,2015-07-01
6,2015,July,1,2015-07-01
7,2015,July,1,2015-07-01


### 3.5 Columnas nuevas

In [15]:
# Fecha de salida
df["fecha_salida"] = df["fecha_llegada"] + pd.to_timedelta(df["total_noches"], unit="D")

# Número total de huéspedes
df["total_huespedes"] = df["adults"] + df["children"] + df["babies"]

# Ingreso de la reserva = precio por noche * noches (solo es real si no se canceló)
df["ingreso_reserva"] = (df["adr"] * df["total_noches"]).round(2)
df["ingreso_real"] = np.where(df["is_canceled"] == 0, df["ingreso_reserva"], 0)

# Día de la semana de llegada (0 = lunes)
dias = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}
df["dia_semana_llegada"] = df["fecha_llegada"].dt.dayofweek.map(dias)

# Temporada según el mes de llegada
def temporada(mes):
    """Devuelve la temporada (Invierno, Primavera, Verano u Otoño) de un mes."""
    if mes in [12, 1, 2]:
        return "Invierno"
    elif mes in [3, 4, 5]:
        return "Primavera"
    elif mes in [6, 7, 8]:
        return "Verano"
    else:
        return "Otoño"

df["temporada"] = df["mes_llegada"].apply(temporada)

# ¿Le dieron la habitación que reservó?
df["habitacion_cambiada"] = df["reserved_room_type"] != df["assigned_room_type"]

# Grupos de antelación (lead_time = días entre la reserva y la llegada)
df["grupo_antelacion"] = pd.cut(df["lead_time"],
                                bins=[-1, 7, 30, 90, 180, 1000],
                                labels=["0-7 días", "8-30 días", "31-90 días", "91-180 días", "Más de 180 días"])

df.shape

(86637, 43)

### 3.6 Traducción de valores y nombres de columnas
Paso los valores principales y los nombres de las columnas a español para que el dashboard y el informe sean más claros.

In [16]:
df["hotel"] = df["hotel"].replace({"City Hotel": "Hotel ciudad (Lisboa)", "Resort Hotel": "Resort (Algarve)"})
df["meal"] = df["meal"].replace({"BB": "Alojamiento y desayuno", "HB": "Media pensión",
                                 "FB": "Pensión completa", "SC": "Solo alojamiento"})
df["deposit_type"] = df["deposit_type"].replace({"No Deposit": "Sin depósito", "Non Refund": "No reembolsable",
                                                 "Refundable": "Reembolsable"})
df["reservation_status"] = df["reservation_status"].replace({"Check-Out": "Completada", "Canceled": "Cancelada",
                                                             "No-Show": "No presentado"})

df = df.rename(columns={
    "is_canceled": "cancelada",
    "lead_time": "antelacion_dias",
    "arrival_date_year": "anio_llegada",
    "arrival_date_week_number": "semana_llegada",
    "arrival_date_day_of_month": "dia_llegada",
    "stays_in_weekend_nights": "noches_fin_semana",
    "stays_in_week_nights": "noches_entre_semana",
    "adults": "adultos",
    "children": "ninos",
    "babies": "bebes",
    "meal": "regimen",
    "country": "codigo_pais",
    "market_segment": "segmento_mercado",
    "distribution_channel": "canal_distribucion",
    "is_repeated_guest": "cliente_repetidor",
    "previous_cancellations": "cancelaciones_previas",
    "previous_bookings_not_canceled": "reservas_previas_no_canceladas",
    "reserved_room_type": "habitacion_reservada",
    "assigned_room_type": "habitacion_asignada",
    "booking_changes": "cambios_reserva",
    "deposit_type": "tipo_deposito",
    "days_in_waiting_list": "dias_lista_espera",
    "customer_type": "tipo_cliente",
    "adr": "precio_medio_noche",
    "required_car_parking_spaces": "plazas_parking",
    "total_of_special_requests": "peticiones_especiales",
    "reservation_status": "estado_reserva",
    "reservation_status_date": "fecha_estado_reserva",
})

# La columna del mes en texto ya no hace falta, tengo mes_llegada en número y fecha_llegada
df = df.drop(columns=["arrival_date_month"])
df.columns.tolist()

['hotel', 'cancelada', 'antelacion_dias', 'anio_llegada', 'semana_llegada', 'dia_llegada', 'noches_fin_semana', 'noches_entre_semana', 'adultos', 'ninos', 'bebes', 'regimen', 'codigo_pais', 'segmento_mercado', 'canal_distribucion', 'cliente_repetidor', 'cancelaciones_previas', 'reservas_previas_no_canceladas', 'habitacion_reservada', 'habitacion_asignada', 'cambios_reserva', 'tipo_deposito', 'dias_lista_espera', 'tipo_cliente', 'precio_medio_noche', 'plazas_parking', 'peticiones_especiales', 'estado_reserva', 'fecha_estado_reserva', 'tiene_agencia', 'reserva_empresa', 'total_noches', 'mes_llegada', 'fecha_llegada', 'fecha_salida', 'total_huespedes', 'ingreso_reserva', 'ingreso_real', 'dia_semana_llegada', 'temporada', 'habitacion_cambiada', 'grupo_antelacion']

## 4. Preparación de los datos del Banco Mundial

In [17]:
pib.head(3)

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6767.559229,8244.045660,10056.261393,11507.217151,12187.536361,13233.990517,13892.605143,14700.959808,16055.287787,16548.717387,16620.954556,17750.009564,18828.087059,19216.197235,20681.023027,20740.132583,21307.248251,21949.485996,23700.631990,24171.837133,24845.658484,26736.308944,28171.909401,25134.771230,24093.140151,25712.384302,25119.665545,25813.571441,26129.839062,27458.220154,27441.550214,28440.041688,30082.158423,30654.485124,22664.370995,26827.344787,31000.571380,34897.618393,38590.565029,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.089515,186.909365,197.367876,225.400456,208.963066,226.836513,240.912196,243.774,257.144252,281.579364,276.733386,294.814654,311.465573,389.729675,463.469549,479.079598,468.775304,518.35955,571.618271,634.449175,773.302496,777.629258,725.548722,732.346684,650.347927,554.222221,578.318417,664.792923,704.143241,728.244534,822.404372,864.175529,732.912139,709.324718,700.773295,766.524211,746.788581,767.400693,697.080579,670.425332,706.728441,625.828860,630.513380,815.433658,989.017115,1126.299204,1235.170879,1381.445251,1447.427775,1408.615659,1628.768573,1761.838296,1731.869096,1705.839003,1689.982381,1498.715971,1335.743165,1529.923080,1553.617944,1508.031525,1351.503167,1560.894626,1675.902524,1571.132704,1628.227289,1722.38562,NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,174.930991,138.706822,178.954088,198.871116,221.763654,254.184249,274.218554,376.223152,381.733238,452.053705,560.621505,606.694676,651.417134,637.087099,625.054942,565.569730,522.082216,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,416.871146,NaN,NaN


In [18]:
# Me quedo solo con los años de las reservas (2015, 2016 y 2017)
pib_anios = pib[["Country Code", "2015", "2016", "2017"]]

# Paso los años de columnas a filas (formato largo) para poder unir por país y año
pib_largo = pib_anios.melt(id_vars="Country Code", var_name="anio_llegada", value_name="pib_per_capita")
pib_largo["anio_llegada"] = pib_largo["anio_llegada"].astype(int)
pib_largo["pib_per_capita"] = pib_largo["pib_per_capita"].round(2)
pib_largo.head()

,Country Code,anio_llegada,pib_per_capita
0,ABW,2015,27458.22
1,AFE,2015,1498.72
2,AFG,2015,565.57
3,AFW,2015,1863.40
4,AGO,2015,3641.73


In [19]:
# En los metadatos, los agregados ("World", "Euro area"...) no tienen región. Me quedo solo con países reales
paises_reales = paises[paises["Region"].notna()][["Country Code", "TableName", "Region", "IncomeGroup"]]

# Traduzco el nivel de renta
paises_reales["IncomeGroup"] = paises_reales["IncomeGroup"].replace({
    "High income": "Renta alta",
    "Upper middle income": "Renta media-alta",
    "Lower middle income": "Renta media-baja",
    "Low income": "Renta baja",
})

paises_reales = paises_reales.rename(columns={"Country Code": "codigo_pais", "TableName": "pais",
                                              "Region": "region_mundial", "IncomeGroup": "nivel_renta"})
pib_largo = pib_largo.rename(columns={"Country Code": "codigo_pais"})

# Uno PIB y metadatos en una sola tabla del Banco Mundial
banco_mundial = pd.merge(pib_largo, paises_reales, on="codigo_pais", how="inner")
print(banco_mundial.shape)
banco_mundial.head()

(651, 6)


,codigo_pais,anio_llegada,pib_per_capita,pais,region_mundial,nivel_renta
0,ABW,2015,27458.22,Aruba,Latin America & Caribbean,Renta alta
1,AFG,2015,565.57,Afghanistan,Middle East & North Africa,Renta baja
2,AGO,2015,3641.73,Angola,Sub-Saharan Africa,Renta media-baja
3,ALB,2015,4199.54,Albania,Europe & Central Asia,Renta media-alta
4,AND,2015,38654.93,Andorra,Europe & Central Asia,Renta alta


## 5. Preparación de los datos del tiempo

In [20]:
# Añado a cada tabla a qué hotel corresponde, con el mismo texto que en las reservas
tiempo_lisboa["hotel"] = "Hotel ciudad (Lisboa)"
tiempo_algarve["hotel"] = "Resort (Algarve)"

# Junto las dos tablas una debajo de la otra
tiempo = pd.concat([tiempo_lisboa, tiempo_algarve], ignore_index=True)

tiempo = tiempo.rename(columns={
    "time": "fecha_llegada",
    "temperature_2m_max (°C)": "temp_max",
    "temperature_2m_min (°C)": "temp_min",
    "precipitation_sum (mm)": "lluvia_mm",
    "sunshine_duration (s)": "sol_segundos",
})

tiempo["fecha_llegada"] = pd.to_datetime(tiempo["fecha_llegada"])
# Paso las horas de sol de segundos a horas, que se entiende mejor
tiempo["horas_sol"] = (tiempo["sol_segundos"] / 3600).round(1)
tiempo = tiempo.drop(columns=["sol_segundos"])
# Marco si llovió ese día (más de 1 mm)
tiempo["dia_lluvioso"] = tiempo["lluvia_mm"] > 1

print(tiempo.shape)
print("Nulos:", tiempo.isna().sum().sum())
tiempo.head()

(1586, 7)
Nulos: 0


,fecha_llegada,temp_max,temp_min,lluvia_mm,hotel,horas_sol,dia_lluvioso
0,2015-07-01,22.6,17.5,0.1,Hotel ciudad (Lisboa),11.2,False
1,2015-07-02,24.1,16.9,0.0,Hotel ciudad (Lisboa),14.3,False
2,2015-07-03,23.9,16.3,0.0,Hotel ciudad (Lisboa),14.5,False
3,2015-07-04,27.0,18.3,0.0,Hotel ciudad (Lisboa),14.2,False
4,2015-07-05,25.5,17.4,0.0,Hotel ciudad (Lisboa),14.3,False


## 6. Unión de las tres fuentes

In [21]:
# Reservas + Banco Mundial por país y año de llegada.
# Uso how="left" para no perder ninguna reserva aunque su país no esté en el Banco Mundial
final = pd.merge(df, banco_mundial, on=["codigo_pais", "anio_llegada"], how="left")

# + tiempo por hotel y fecha de llegada (el tiempo que hacía el día que llegó el cliente)
final = pd.merge(final, tiempo, on=["hotel", "fecha_llegada"], how="left")

print("Filas antes de unir:", len(df))
print("Filas después de unir:", len(final))   # deben ser las mismas: cada reserva solo cruza con una fila

Filas antes de unir: 86637
Filas después de unir: 86637


In [22]:
# Reservas que no han encontrado su país en el Banco Mundial
sin_pais = final["pais"].isna()
print("Reservas sin datos del Banco Mundial:", sin_pais.sum())
final.loc[sin_pais, "codigo_pais"].value_counts()

Reservas sin datos del Banco Mundial: 503


codigo_pais
Desconocido    442
TWN             43
JEY              8
GGY              2
MYT              2
ATA              2
GLP              1
UMI              1
AIA              1
ATF              1
Name: count, dtype: int64

Las reservas sin datos del Banco Mundial son las de país desconocido y algunos territorios que el Banco Mundial no recoge
(Taiwán, Jersey, Guernsey...). Son muy pocas, así que las mantengo y relleno sus textos como "Desconocido".
El PIB lo dejo vacío, porque inventarlo falsearía el análisis.

In [23]:
final["pais"] = final["pais"].fillna("Desconocido")
final["region_mundial"] = final["region_mundial"].fillna("Desconocido")
final["nivel_renta"] = final["nivel_renta"].fillna("Desconocido")

# Comprobación: el tiempo tiene que haber cruzado con todas las reservas
print("Reservas sin datos del tiempo:", final["temp_max"].isna().sum())

Reservas sin datos del tiempo: 0


## 7. Comprobación de los requisitos mínimos

In [24]:
filas, columnas = final.shape
print(f"Conjunto final: {filas} filas y {columnas} columnas")
print("¿Cumple 50.000 filas?", filas >= 50000)
print("¿Cumple 20 columnas?", columnas >= 20)

Conjunto final: 86637 filas y 51 columnas
¿Cumple 50.000 filas? True
¿Cumple 20 columnas? True


In [25]:
final.info()

<class 'pandas.DataFrame'>
RangeIndex: 86637 entries, 0 to 86636
Data columns (total 51 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   hotel                           86637 non-null  str           
 1   cancelada                       86637 non-null  int64         
 2   antelacion_dias                 86637 non-null  int64         
 3   anio_llegada                    86637 non-null  int64         
 4   semana_llegada                  86637 non-null  int64         
 5   dia_llegada                     86637 non-null  int64         
 6   noches_fin_semana               86637 non-null  int64         
 7   noches_entre_semana             86637 non-null  int64         
 8   adultos                         86637 non-null  int64         
 9   ninos                           86637 non-null  int64         
 10  bebes                           86637 non-null  int64         
 11  regimen      

## 8. Guardado del conjunto final

In [26]:
import os
os.makedirs(RUTA_PROCESSED, exist_ok=True)   # crea la carpeta processed si no existe
final.to_csv(RUTA_PROCESSED + "hotel_bookings_final.csv", index=False)
print("Guardado en", RUTA_PROCESSED + "hotel_bookings_final.csv")

Guardado en ../data/processed/hotel_bookings_final.csv


## 9. Tablas para el modelo en estrella de Power BI
Además del conjunto final (que cumple el requisito de 50.000 filas y 20 columnas), guardo unas tablas de dimensiones
para montar en Power BI un **modelo en estrella**: una tabla de hechos (las reservas) unida a tablas de dimensiones
(países, hoteles y tiempo). Así no se repite información y el modelo ocupa menos.

In [27]:
# Dimensión países: una fila por código de país con los datos del Banco Mundial
dim_paises = (final[["codigo_pais", "pais", "region_mundial", "nivel_renta"]]
              .drop_duplicates(subset="codigo_pais")
              .sort_values("codigo_pais"))

# Dimensión hoteles: una fila por hotel
dim_hoteles = pd.DataFrame({
    "hotel": ["Hotel ciudad (Lisboa)", "Resort (Algarve)"],
    "tipo_hotel": ["Hotel urbano", "Resort de playa"],
    "ciudad": ["Lisboa", "Faro"],
    "region": ["Lisboa", "Algarve"],
})

# Dimensión tiempo: una fila por hotel y día. Creo una clave única hotel|fecha para relacionarla con las reservas
dim_tiempo = tiempo.copy()
dim_tiempo["clave_tiempo"] = dim_tiempo["hotel"] + "|" + dim_tiempo["fecha_llegada"].dt.strftime("%Y-%m-%d")
dim_tiempo = dim_tiempo.rename(columns={"fecha_llegada": "fecha"})
dim_tiempo = dim_tiempo[["clave_tiempo", "hotel", "fecha", "temp_max", "temp_min", "lluvia_mm", "horas_sol", "dia_lluvioso"]]

# Comprobación: las claves tienen que ser únicas en las dimensiones
print("Países repetidos:", dim_paises["codigo_pais"].duplicated().sum())
print("Claves de tiempo repetidas:", dim_tiempo["clave_tiempo"].duplicated().sum())

dim_paises.to_csv(RUTA_PROCESSED + "dim_paises.csv", index=False)
dim_hoteles.to_csv(RUTA_PROCESSED + "dim_hoteles.csv", index=False)
dim_tiempo.to_csv(RUTA_PROCESSED + "dim_tiempo.csv", index=False)
print(dim_paises.shape, dim_hoteles.shape, dim_tiempo.shape)

Países repetidos: 0
Claves de tiempo repetidas: 0
(177, 4) (2, 4) (1586, 8)
